# CI/CD & Production Workflows

Hardcoding comparison rules directly into application logic creates maintenance bottlenecks in production environments. This tutorial demonstrates how to decouple **Policy** (validation rules) from **Execution** (pipeline code) using Veridelta's YAML configuration.

### 1. The GitOps Validation Pattern

Veridelta enables a declarative, GitOps-driven workflow for data pipeline validation:
1. **Store:** Maintain `veridelta.yaml` in version control alongside your dbt models or Airflow DAGs.
2. **Execute:** Trigger a generic validation script during the CI/CD pipeline.
3. **Update:** Modify comparison tolerances via Pull Requests without altering the core Python orchestration code.

In [ ]:
import polars as pl

# Mock production data (e.g., Nightly User LTV Pipeline)
src_df = pl.DataFrame(
    {
        "user_id": ["U-100", "U-101"],
        "ltv_score": [850.50, 420.00],
        "cohort": ["Q1_Active", "Q2_Churned"],
    }
)

# Simulate downstream drift ($0.05 drift; lowercase strings)
tgt_df = pl.DataFrame(
    {
        "user_id": ["U-100", "U-101"],
        "ltv_score": [850.55, 420.00],
        "cohort": ["q1_active", "q2_churned"],
    }
)

# Simulate remote storage artifacts (S3/GCS) via local disk
src_df.write_parquet("prod_source.parquet")
tgt_df.write_parquet("prod_target.parquet")

print("SYSTEM: Production mock artifacts generated.")

# Output:
# SYSTEM: Production mock artifacts generated.

### 2. Define the Declarative Policy

The YAML configuration file dictates the execution parameters. Because Veridelta uses strict Pydantic schemas, this file is fully validated for structural integrity before any expensive I/O operations occur.

In [ ]:
%%writefile veridelta.yaml
primary_keys: ["user_id"]
default_absolute_tolerance: 0.10

source:
  path: "prod_source.parquet"
  format: "parquet"

target:
  path: "prod_target.parquet"
  format: "parquet"

rules:
  - column_names: ["cohort"]
    case_insensitive: true

### 3. Generic Execution Script

Pipeline orchestrators (e.g., GitHub Actions, Airflow, Dagster) utilize a generic execution script. By using the `engine.execute()` API, this script remains fully agnostic to schemas, tolerances, and formats—delegating all lazy data ingestion and validation logic to the YAML configuration.

In [ ]:
from veridelta.config import load_config
from veridelta.engine import DiffEngine


def run_pipeline_validation(config_path: str) -> None:
    """Load configuration and execute the semantic diff pipeline end-to-end."""
    # 1. Parse and strictly validate the YAML policy
    diff_cfg, src_cfg, tgt_cfg = load_config(config_path)

    # 2. Initialize the stateless engine and execute the lazy computation graph
    engine = DiffEngine(diff_cfg)
    summary = engine.execute(src_cfg, tgt_cfg)

    # 3. Output standard Markdown reporting
    print(summary.report_summary)

    # 4. Enforce the deployment gate
    if not summary.is_match:
        # In production CI/CD, raise an exception to block the deployment:
        raise AssertionError(
            f"Data regression detected! {summary.changed_count} rows failed validation."
        )


run_pipeline_validation("veridelta.yaml")

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        PASSED (Perfect Match)
# Match Rate:    100.0%
# Source Rows:   2
# Target Rows:   2
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       0
# Total Issues:  0

### Advanced: Environment Overrides

During environment promotion (e.g., Staging to Production), the `SourceConfig.options` block in your Python script accepts dynamic cloud credentials or environment-specific path overrides. This ensures your semantic validation rules remain strictly identical across all deployment tiers, even if the underlying storage layer changes.

In [ ]:
import pathlib

# Remove local artifacts (housekeeping)
for f in ["prod_source.parquet", "prod_target.parquet", "veridelta.yaml"]:
    pathlib.Path(f).unlink(missing_ok=True)

print("SYSTEM: Local artifacts cleaned.")

# Output:
# SYSTEM: Local artifacts cleaned.